# Qwen-2.5 1.5B Instruct Model - Kaggle Setup and Usage

This notebook demonstrates how to load, configure, and use the Qwen-2.5 1.5B Instruct model in Kaggle environment with GPU acceleration and memory optimization.

## Features:
- 🚀 Optimized model loading with 4-bit quantization
- 🎭 Proper chat template formatting
- 🔄 Batch processing capabilities  
- 💾 Memory management utilities
- 🎯 Real-world usage examples
- 📊 Performance monitoring

## Model Information:
- **Model**: Qwen/Qwen2.5-1.5B-Instruct
- **Size**: 1.5B parameters
- **Type**: Instruction-tuned conversational model
- **Framework**: Transformers + PyTorch

## 1. Environment Setup and Package Installation

In [ ]:
# Install required packages for Kaggle environment
!pip install transformers torch accelerate bitsandbytes -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install flash-attn --no-build-isolation -q

print("✅ All packages installed successfully!")

## 2. Imports and Configuration

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import warnings
import json
import os
import gc
import time
from typing import List, Dict, Optional
from tqdm.auto import tqdm
import pandas as pd

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Device configuration and system info
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Using device: {device}")
print(f"⚡ CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"🎯 GPU: {torch.cuda.get_device_name()}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"🔢 CUDA Version: {torch.version.cuda}")

# Global configuration for Qwen-2.5 1.5B Instruct
CONFIG = {
    "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
    "max_new_tokens": 512,
    "temperature": 0.7,
    "top_p": 0.9,
    "do_sample": True,
    "use_cache": True,
    "device": "cuda:0" if torch.cuda.is_available() else "cpu",
    "use_4bit": True,
    "use_flash_attention": True
}

print(f"\n✅ Configuration loaded for {CONFIG['model_name']}")
print(f"📊 Max tokens: {CONFIG['max_new_tokens']}")
print(f"🌡️ Temperature: {CONFIG['temperature']}")

/Users/virat_garg/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/virat_garg/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🖥️  Using device: cpu
⚡ CUDA available: False

✅ Configuration loaded for Qwen/Qwen2.5-1.5B-Instruct
📊 Max tokens: 512
🌡️ Temperature: 0.7


## 3. Model and Tokenizer Loading

In [2]:
def load_qwen_model_and_tokenizer():
    """
    Load Qwen-2.5 1.5B Instruct model with optimizations for Kaggle environment.
    """
    
    print(f"🚀 Loading {CONFIG['model_name']}...")
    start_time = time.time()
    
    # Configure 4-bit quantization for memory efficiency
    if CONFIG["use_4bit"] and torch.cuda.is_available():
        print("🔧 Configuring 4-bit quantization for memory efficiency...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )
    else:
        bnb_config = None
        print("⚠️ Using full precision (no quantization)")
    
    try:
        # Load tokenizer first
        print("📝 Loading tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(
            CONFIG["model_name"],
            trust_remote_code=True,
            padding_side="left"
        )
        
        # Configure padding token
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            tokenizer.pad_token_id = tokenizer.eos_token_id
        
        print(f"   ✅ Tokenizer loaded - Vocab size: {len(tokenizer)}")
        
        # Prepare model loading arguments
        model_kwargs = {
            "trust_remote_code": True,
            "low_cpu_mem_usage": True,
        }
        
        # Configure data type based on device
        if torch.cuda.is_available():
            model_kwargs["torch_dtype"] = torch.bfloat16
        else:
            model_kwargs["torch_dtype"] = torch.float32
        
        # Add quantization if enabled
        if bnb_config is not None:
            model_kwargs["quantization_config"] = bnb_config
            model_kwargs["device_map"] = "auto"
        
        # Add flash attention if requested and available
        if CONFIG["use_flash_attention"] and torch.cuda.is_available():
            try:
                model_kwargs["attn_implementation"] = "flash_attention_2"
                print("⚡ Using Flash Attention 2 for faster inference")
            except:
                print("⚠️ Flash Attention 2 not available, using default attention")
        
        # Load the model
        print("🧠 Loading model with optimizations...")
        model = AutoModelForCausalLM.from_pretrained(
            CONFIG["model_name"],
            **model_kwargs
        )
        
        # Move to device if not using quantization device_map
        if bnb_config is None:
            model = model.to(device)
        
        model.eval()  # Set to evaluation mode
        
        load_time = time.time() - start_time
        print(f"✅ Model loaded successfully in {load_time:.2f} seconds!")
        
        # Display model information
        print(f"\n📊 Model Information:")
        try:
            param_count = sum(p.numel() for p in model.parameters())
            print(f"   📈 Parameters: {param_count / 1e9:.2f}B")
        except:
            print(f"   📈 Parameters: ~1.5B (estimated)")
        
        print(f"   🎯 Model device: {next(model.parameters()).device}")
        print(f"   🔢 Model dtype: {next(model.parameters()).dtype}")
        
        # Memory usage info
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1e9
            cached = torch.cuda.memory_reserved() / 1e9
            print(f"   💾 GPU Memory - Allocated: {allocated:.2f}GB, Reserved: {cached:.2f}GB")
        
        return model, tokenizer
        
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        raise

# Load the model and tokenizer
model, tokenizer = load_qwen_model_and_tokenizer()

🚀 Loading Qwen/Qwen2.5-1.5B-Instruct...
⚠️ Using full precision (no quantization)
📝 Loading tokenizer...
   ✅ Tokenizer loaded - Vocab size: 151665
🧠 Loading model with optimizations...


`torch_dtype` is deprecated! Use `dtype` instead!


✅ Model loaded successfully in 282.37 seconds!

📊 Model Information:
   📈 Parameters: 1.54B
   🎯 Model device: cpu
   🔢 Model dtype: torch.float32


## 4. Qwen Chat Generator Class

In [4]:
class QwenChatGenerator:
    """
    Advanced wrapper for Qwen-2.5 1.5B Instruct model with proper chat formatting.
    """
    
    def __init__(self, model, tokenizer, device="cuda"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.generation_count = 0
        print(f"🎭 QwenChatGenerator initialized on {device}")
    
    def format_chat_prompt(self, user_message: str, system_message: Optional[str] = None) -> str:
        """
        Format messages using Qwen's official chat template.
        """
        messages = []
        
        # Add system message if provided
        if system_message:
            messages.append({"role": "system", "content": system_message})
        
        # Add user message
        messages.append({"role": "user", "content": user_message})
        
        # Apply Qwen's chat template
        try:
            formatted_prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            return formatted_prompt
        except Exception as e:
            # Fallback formatting if chat template fails
            print(f"⚠️ Chat template failed, using fallback: {e}")
            if system_message:
                return f"System: {system_message}\nUser: {user_message}\nAssistant: "
            else:
                return f"User: {user_message}\nAssistant: "
    
    def generate_response(self, 
                         user_message: str, 
                         system_message: Optional[str] = None,
                         max_new_tokens: int = None,
                         temperature: float = None,
                         top_p: float = None,
                         do_sample: bool = None,
                         repetition_penalty: float = 1.1) -> str:
        """
        Generate a single response using the Qwen model.
        """
        
        # Use config defaults if parameters not specified
        max_new_tokens = max_new_tokens or CONFIG["max_new_tokens"]
        temperature = temperature or CONFIG["temperature"]
        top_p = top_p or CONFIG["top_p"]
        do_sample = do_sample if do_sample is not None else CONFIG["do_sample"]
        
        try:
            # Format the chat prompt
            formatted_prompt = self.format_chat_prompt(user_message, system_message)
            
            # Tokenize input
            inputs = self.tokenizer(
                formatted_prompt,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=2048
            )
            
            # Move inputs to correct device
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            # Generate response with optimized parameters
            with torch.no_grad():
                outputs = self.model.generate(
                    inputs['input_ids'],
                    attention_mask=inputs.get('attention_mask'),
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    top_p=top_p,
                    do_sample=do_sample,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    repetition_penalty=repetition_penalty,
                    no_repeat_ngram_size=3
                )
            
            # Extract only the generated tokens
            input_length = inputs['input_ids'].shape[1]
            response_tokens = outputs[0][input_length:]
            response = self.tokenizer.decode(response_tokens, skip_special_tokens=True)
            
            self.generation_count += 1
            return response.strip()
            
        except Exception as e:
            print(f"❌ Generation error: {e}")
            return f"Generation failed: {str(e)[:100]}"
    
    def batch_generate(self, 
                      messages: List[str], 
                      system_message: Optional[str] = None,
                      max_new_tokens: int = None,
                      batch_size: int = 2,
                      show_progress: bool = True) -> List[str]:
        """
        Generate responses for multiple messages efficiently.
        """
        
        max_new_tokens = max_new_tokens or CONFIG["max_new_tokens"]
        responses = []
        
        iterator = range(0, len(messages), batch_size)
        if show_progress:
            iterator = tqdm(iterator, desc="Generating responses", unit="batch")
        
        for i in iterator:
            batch_messages = messages[i:i + batch_size]
            batch_responses = []
            
            for message in batch_messages:
                response = self.generate_response(
                    user_message=message,
                    system_message=system_message,
                    max_new_tokens=max_new_tokens
                )
                batch_responses.append(response)
            
            responses.extend(batch_responses)
            
            # Periodic memory cleanup
            if torch.cuda.is_available() and (i // batch_size) % 5 == 0:
                torch.cuda.empty_cache()
        
        return responses
    
    def get_stats(self) -> Dict:
        """Get generation statistics."""
        return {
            "total_generations": self.generation_count,
            "model_name": CONFIG["model_name"],
            "device": str(self.device)
        }

# Initialize the generator
print("🎭 Initializing Qwen Chat Generator...")
generator = QwenChatGenerator(model, tokenizer, device=CONFIG["device"])
print("✅ Generator ready for use!")

🎭 Initializing Qwen Chat Generator...
🎭 QwenChatGenerator initialized on cpu
✅ Generator ready for use!


## 5. Testing and Examples

In [5]:
# Test 1: Basic conversation
print("🧪 Test 1: Basic Conversation")
print("=" * 50)

response = generator.generate_response(
    user_message="What are the main advantages of transformer models in NLP?",
    system_message="You are a helpful AI assistant expert in machine learning and NLP.",
    max_new_tokens=200
)

print("💬 User: What are the main advantages of transformer models in NLP?")
print(f"🤖 Assistant: {response}")
print()

# Test 2: Creative writing
print("🧪 Test 2: Creative Writing")
print("=" * 50)

response = generator.generate_response(
    user_message="Write a short story about a robot learning to paint.",
    system_message="You are a creative writing assistant. Write engaging and imaginative stories.",
    max_new_tokens=300,
    temperature=0.8
)

print("💬 User: Write a short story about a robot learning to paint.")
print(f"🤖 Assistant: {response}")
print()

# Test 3: Code generation
print("🧪 Test 3: Code Generation")
print("=" * 50)

response = generator.generate_response(
    user_message="Write a Python function to calculate the Fibonacci sequence using recursion.",
    system_message="You are a programming assistant. Provide clean, well-commented code.",
    max_new_tokens=250,
    temperature=0.3
)

print("💬 User: Write a Python function to calculate the Fibonacci sequence using recursion.")
print(f"🤖 Assistant: {response}")

🧪 Test 1: Basic Conversation
💬 User: What are the main advantages of transformer models in NLP?
🤖 Assistant: Transformer models have several key advantages in Natural Language Processing (NLP) that set them apart from earlier sequence-to-sequence approaches like recurrent neural networks (RNNs). Here are some of the most significant advantages:

1. **Attention Mechanism**: The primary innovation of transformers is their attention mechanism, which allows the model to focus on different parts of the input sequence during processing. This makes it more efficient at handling long-range dependencies compared to RNNs.

2. **Efficient Memory Usage**: Transformer architectures typically use less memory than sequential models because they don't need to maintain an internal state for each word in the sequence. Instead, they only store the context needed to make predictions for each token.

3. **Scalability**: Transformers can scale well with increasing data sizes, making them suitable for large 

## 6. Batch Processing Example

In [ ]:
# Batch processing example
print("🔄 Testing Batch Processing")
print("=" * 50)

# Sample questions for batch processing
questions = [
    "What is machine learning?",
    "Explain the concept of neural networks.",
    "What is the difference between AI and ML?",
    "How do transformers work in NLP?",
    "What is the purpose of attention mechanisms?"
]

system_instruction = "You are an AI tutor. Provide clear, concise explanations suitable for beginners."

print(f"📝 Processing {len(questions)} questions in batch...")
start_time = time.time()

batch_responses = generator.batch_generate(
    messages=questions,
    system_message=system_instruction,
    max_new_tokens=150,
    batch_size=2
)

batch_time = time.time() - start_time
print(f"⏱️ Batch processing completed in {batch_time:.2f} seconds")
print(f"📊 Average time per question: {batch_time/len(questions):.2f} seconds")

# Display results
for i, (question, answer) in enumerate(zip(questions, batch_responses), 1):
    print(f"\n💬 Question {i}: {question}")
    print(f"🤖 Answer: {answer}")
    print("-" * 80)

## 7. Emotion Analysis Example

In [8]:
def analyze_emotions_with_qwen(text: str) -> Dict:
    """
    Use Qwen to analyze emotions in text with structured output.
    """
    
    system_prompt = """You are an expert emotion analyst. Analyze the given text and provide emotion scores for anger, disgust, and contempt on a scale of 0-100.

Return your analysis in this exact JSON format (no additional text):
{
    "anger": <score 0-100>,
    "disgust": <score 0-100>, 
    "contempt": <score 0-100>,
    "happy": <score 0-100>,
    "sad": <score 0-100>,
    "explanation": "<brief explanation of the emotional content>"
}"""
    
    user_prompt = f"Analyze the emotional content of this text: '{text}'"
    
    response = generator.generate_response(
        user_message=user_prompt,
        system_message=system_prompt,
        max_new_tokens=200,
        temperature=0.2  # Lower temperature for consistent structured output
    )
    
    # Parse JSON response
    try:
        import re
        # Extract JSON from response
        json_match = re.search(r'\{[^{}]*\}', response, re.DOTALL)
        if json_match:
            json_str = json_match.group()
            result = json.loads(json_str)
            result["raw_response"] = response
            return result
    except:
        pass
    
    # Fallback if JSON parsing fails
    return {
        "anger": 0,
        "disgust": 0,
        "contempt": 0,
        "explanation": "Could not parse structured response",
        "raw_response": response
    }

# Test emotion analysis
print("🎭 Testing Emotion Analysis")
print("=" * 50)

test_texts = [
    "I absolutely love this new technology! It's amazing!",
    "I can't believe how stupid and worthless that decision was.",
    "The weather is nice today and I had a good lunch.",
    "You people are beneath my level and don't deserve my attention."
]

emotion_results = []

for i, text in enumerate(test_texts, 1):
    print(f"\n📝 Text {i}: '{text}'")
    
    result = analyze_emotions_with_qwen(text)
    emotion_results.append({"text": text, **result})
    
    print(f"😠 Anger: {result['anger']}/100")
    print(f"🤢 Disgust: {result['disgust']}/100") 
    print(f"😤 Contempt: {result['contempt']}/100")
    print(f"💭 Explanation: {result['explanation']}")
    print(f" Raw Response: {result['raw_response'][:100]}...")  # Show first 100 chars of raw response

# Create summary DataFrame
df_emotions = pd.DataFrame(emotion_results)
print("\n📊 Emotion Analysis Summary:")
print(df_emotions[['text', 'anger', 'disgust', 'contempt']].to_string(index=False))

🎭 Testing Emotion Analysis

📝 Text 1: 'I absolutely love this new technology! It's amazing!'
😠 Anger: 0/100
🤢 Disgust: 0/100
😤 Contempt: 0/100
💭 Explanation: Could not parse structured response
 Raw Response: ```json
{
    'anger': 0,
    'disgusted': 25,
    "Contempt": 0,  
    "happiness": 98,
    // Expl...

📝 Text 2: 'I can't believe how stupid and worthless that decision was.'
😠 Anger: 0/100
🤢 Disgust: 0/100
😤 Contempt: 0/100
💭 Explanation: Could not parse structured response
 Raw Response: ```json
{
    'anger': 95,
    'disgusted': 45,
}
```

**Explanation:** The text expresses strong ne...

📝 Text 3: 'The weather is nice today and I had a good lunch.'
😠 Anger: 0/100
🤢 Disgust: 0/100
😤 Contempt: 0/100
💭 Explanation: Could not parse structured response
 Raw Response: ```json
{
    'anger': 0,
    'disgusted': 25,
    "Contempt": 0, 
    'happy': 85,
     'sad': 30,
...

📝 Text 4: 'You people are beneath my level and don't deserve my attention.'
😠 Anger: 0/100
🤢 Disgust: 0/100
😤 

## 8. Memory Management and Performance Monitoring

In [ ]:
def get_memory_usage():
    """Monitor GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        
        print(f"🔍 GPU Memory Status:")
        print(f"   📈 Allocated: {allocated:.2f}GB")
        print(f"   📦 Reserved: {reserved:.2f}GB") 
        print(f"   💾 Total: {total:.2f}GB")
        print(f"   📊 Usage: {(allocated/total)*100:.1f}%")
        
        return allocated, reserved, total
    else:
        print("❌ CUDA not available")
        return 0, 0, 0

def clear_memory():
    """Clear GPU memory cache."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        print("🧹 Memory cache cleared")

def benchmark_generation(num_tests: int = 5):
    """Benchmark generation speed."""
    print(f"⏱️ Benchmarking generation speed ({num_tests} tests)...")
    
    test_prompt = "Explain the concept of artificial intelligence in simple terms."
    times = []
    
    for i in range(num_tests):
        start_time = time.time()
        response = generator.generate_response(
            user_message=test_prompt,
            max_new_tokens=100
        )
        generation_time = time.time() - start_time
        times.append(generation_time)
        
        tokens_generated = len(generator.tokenizer.encode(response))
        tokens_per_sec = tokens_generated / generation_time
        print(f"   Test {i+1}: {generation_time:.2f}s ({tokens_per_sec:.1f} tokens/sec)")
    
    avg_time = sum(times) / len(times)
    print(f"📊 Average generation time: {avg_time:.2f} seconds")
    
    return times

# Monitor current memory usage
print("📊 Current System Status:")
get_memory_usage()
print()

# Show generation statistics
stats = generator.get_stats()
print("📈 Generation Statistics:")
for key, value in stats.items():
    print(f"   {key}: {value}")
print()

# Run benchmark
benchmark_times = benchmark_generation(3)

# Clear memory
print()
clear_memory()
get_memory_usage()

## 9. Advanced Usage Examples

In [ ]:
# Advanced Example 1: Multi-turn conversation simulation
print("💬 Advanced Example 1: Multi-turn Conversation")
print("=" * 60)

conversation_history = []
system_msg = "You are a helpful tutor teaching about machine learning. Keep track of the conversation context."

topics = [
    "What is supervised learning?",
    "Can you give me an example of supervised learning?",
    "How is this different from unsupervised learning?",
    "What algorithms would you recommend for a beginner?"
]

for i, topic in enumerate(topics, 1):
    print(f"\n🗣️ Turn {i}: {topic}")
    
    response = generator.generate_response(
        user_message=topic,
        system_message=system_msg,
        max_new_tokens=200
    )
    
    conversation_history.append({"user": topic, "assistant": response})
    print(f"🤖 Response: {response}")

print("\n" + "="*60)

# Advanced Example 2: Structured data extraction
print("\n🔍 Advanced Example 2: Structured Data Extraction")
print("=" * 60)

text_to_analyze = """
John Smith works as a Software Engineer at TechCorp Inc. 
He has 5 years of experience and specializes in Python and machine learning. 
His email is john.smith@techcorp.com and he lives in San Francisco, CA.
"""

extraction_prompt = f"""Extract structured information from this text in JSON format:

Text: {text_to_analyze}

Extract: name, job_title, company, experience_years, skills, email, location

Return only the JSON object."""

structured_response = generator.generate_response(
    user_message=extraction_prompt,
    system_message="You are a data extraction specialist. Return only valid JSON.",
    max_new_tokens=150,
    temperature=0.1
)

print(f"📄 Original text: {text_to_analyze}")
print(f"📊 Extracted data: {structured_response}")

# Advanced Example 3: Code review and improvement
print("\n🔧 Advanced Example 3: Code Review")
print("=" * 60)

code_to_review = '''
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
'''

review_response = generator.generate_response(
    user_message=f"Review this Python code and suggest improvements:\n\n{code_to_review}",
    system_message="You are a senior software engineer. Provide detailed code reviews with specific suggestions.",
    max_new_tokens=300
)

print(f"💻 Code to review:\n{code_to_review}")
print(f"📝 Review: {review_response}")

## 10. Model Persistence and Cleanup

In [ ]:
def save_model_for_later_use(save_path: str = "/kaggle/working/qwen_model"):
    """
    Save the model and tokenizer for later use in Kaggle.
    Useful for preserving model state between notebook runs.
    """
    try:
        print(f"💾 Saving model to {save_path}...")
        os.makedirs(save_path, exist_ok=True)
        
        # Save model and tokenizer
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        
        # Save configuration
        config_path = os.path.join(save_path, "config.json")
        with open(config_path, 'w') as f:
            json.dump(CONFIG, f, indent=2)
        
        print(f"✅ Model saved successfully!")
        print(f"   📁 Model files: {save_path}")
        print(f"   ⚙️ Config saved: {config_path}")
        
        # Show saved files
        saved_files = os.listdir(save_path)
        print(f"   📋 Files saved: {len(saved_files)} files")
        
        return save_path
        
    except Exception as e:
        print(f"❌ Error saving model: {e}")
        return None

def load_saved_model(load_path: str = "/kaggle/working/qwen_model"):
    """Load a previously saved model."""
    try:
        if not os.path.exists(load_path):
            print(f"❌ Model path not found: {load_path}")
            return None, None
        
        print(f"📂 Loading model from {load_path}...")
        
        # Load model and tokenizer
        loaded_model = AutoModelForCausalLM.from_pretrained(load_path)
        loaded_tokenizer = AutoTokenizer.from_pretrained(load_path)
        
        print(f"✅ Model loaded successfully from {load_path}")
        return loaded_model, loaded_tokenizer
        
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return None, None

# Final summary and cleanup options
print("🎯 Final Summary")
print("=" * 50)

# Show final statistics
final_stats = generator.get_stats()
print("📊 Session Statistics:")
for key, value in final_stats.items():
    print(f"   {key}: {value}")

# Memory usage summary
print("\n💾 Memory Usage:")
get_memory_usage()

# Save model option (uncomment to use)
print("\n💾 Model Persistence:")
print("   To save model for later use, uncomment the next line:")
print("   # save_path = save_model_for_later_use()")

# save_path = save_model_for_later_use()  # Uncomment to save

print("\n✅ Qwen-2.5 1.5B Instruct setup and testing completed!")
print("🎉 The model is ready for your specific use cases!")

print(f"""
📚 Quick Usage Reference:
   
   # Basic generation
   response = generator.generate_response("Your question here")
   
   # With system message
   response = generator.generate_response(
       user_message="Your question",
       system_message="You are a helpful assistant"
   )
   
   # Batch processing
   responses = generator.batch_generate(["Q1", "Q2", "Q3"])
   
   # Custom parameters
   response = generator.generate_response(
       user_message="Your question",
       max_new_tokens=200,
       temperature=0.8
   )
""")